In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)

!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d meetnagadia/kvasir-dataset
!unzip kvasir-dataset.zip -d dataset

In [ ]:
# ================================
# INSTALL
# ================================
!pip install -q tensorflow scikit-learn xgboost timm torch

import os, cv2, joblib, numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre

import torch
import timm

# ================================
# DATASET
# ================================
DATASET_PATH = "/content/dataset/kvasir-dataset"
LIMIT_PER_CLASS = 500

# ================================
# LOAD MODELS
# ================================
resnet = ResNet50(weights='imagenet', include_top=False, pooling='avg')
efficient = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')

vit = timm.create_model('vit_base_patch16_224', pretrained=True)
vit.eval()

# ================================
# FEATURE EXTRACTION
# ================================
def extract_features(img):

    # CNN inputs
    img_r = cv2.resize(img, (224,224))
    img_e = cv2.resize(img, (224,224))

    r = res_pre(img_r)
    e = eff_pre(img_e)

    r = np.expand_dims(r, axis=0)
    e = np.expand_dims(e, axis=0)

    f1 = resnet.predict(r, verbose=0)
    f2 = efficient.predict(e, verbose=0)

    # ViT input
    img_v = cv2.resize(img, (224,224))
    img_v = img_v.astype(np.float32) / 255.0
    img_v = np.transpose(img_v, (2,0,1))  # HWC → CHW
    img_v = torch.tensor(img_v).unsqueeze(0)

    with torch.no_grad():
        f3 = vit.forward_features(img_v)
        f3 = f3.mean(dim=1).numpy()

    # 🔥 FEATURE FUSION
    fused = np.concatenate([f1, f2, f3], axis=1)

    return fused.flatten()

# ================================
# LOAD DATA
# ================================
X, y = [], []
label_map = {}
label_id = 0

for class_name in os.listdir(DATASET_PATH):

    path = os.path.join(DATASET_PATH, class_name)
    if not os.path.isdir(path): continue

    label_map[label_id] = class_name
    images = os.listdir(path)[:LIMIT_PER_CLASS]

    print(f"Processing {class_name}")

    for img_name in tqdm(images):
        img = cv2.imread(os.path.join(path, img_name))
        if img is None: continue
        try:
            X.append(extract_features(img))
            y.append(label_id)
        except:
            continue

    label_id += 1

X = np.array(X)
y = np.array(y)

# ================================
# SPLIT
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ================================
# SCALING
# ================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ================================
# PCA
# ================================
pca = PCA(n_components=300)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

# ================================
# CLASSIFIERS (TRAIN ALL)
# ================================
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

svm = SVC(kernel='rbf', probability=True)
rf = RandomForestClassifier(n_estimators=200)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    eval_metric='mlogloss'
)

svm.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# ================================
# EVALUATION
# ================================
def evaluate(model, name):
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name} Accuracy:", acc)
    return acc

acc_svm = evaluate(svm, "SVM")
acc_rf = evaluate(rf, "RF")
acc_xgb = evaluate(xgb, "XGBoost")

# ================================
# SELECT BEST MODEL
# ================================
best_model = xgb
if acc_svm > acc_xgb and acc_svm > acc_rf:
    best_model = svm
elif acc_rf > acc_xgb:
    best_model = rf

# ================================
# SAVE
# ================================
joblib.dump(best_model, "final_model.pkl")
joblib.dump(label_map, "labels.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(pca, "pca.pkl")

# ================================
# DOWNLOAD
# ================================
from google.colab import files
files.download("final_model.pkl")
files.download("labels.pkl")
files.download("scaler.pkl")
files.download("pca.pkl")